In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GCNConv

dataset = KarateClub()
data = dataset[0]

print(data)
print("节点数量:", data.num_nodes)
print("边数量:", data.num_edges)
print("类别数量:", dataset.num_classes)


class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # 核心是这两层：节点特征->从邻居聚合信息->更新节点表示->预测每个节点属于哪个类别
        self.conv1 = GCNConv(dataset.num_features, 16)
        self.conv2 = GCNConv(16, dataset.num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x


model = GCN()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4
)

for epoch in range(201):
    model.train()

    optimizer.zero_grad()

    out = model(data.x, data.edge_index)

    loss = F.cross_entropy(out, data.y)

    loss.backward()
    optimizer.step()

    pred = out.argmax(dim=1)
    acc = (pred == data.y).sum() / data.num_nodes

    if epoch % 20 == 0:
        print(
            f"Epoch {epoch:03d}, "
            f"Loss: {loss:.4f}, "
            f"Acc: {acc:.4f}"
        )


model.eval()
out = model(data.x, data.edge_index)
pred = out.argmax(dim=1)

print("真实标签:")
print(data.y)

print("预测标签:")
print(pred)

Data(x=[34, 34], edge_index=[2, 156], y=[34], train_mask=[34])
节点数量: 34
边数量: 156
类别数量: 4
Epoch 000, Loss: 1.3656, Acc: 0.1765
Epoch 020, Loss: 0.8395, Acc: 0.7941
Epoch 040, Loss: 0.3868, Acc: 0.9118
Epoch 060, Loss: 0.1714, Acc: 1.0000
Epoch 080, Loss: 0.0954, Acc: 1.0000
Epoch 100, Loss: 0.0604, Acc: 1.0000
Epoch 120, Loss: 0.0419, Acc: 1.0000
Epoch 140, Loss: 0.0318, Acc: 1.0000
Epoch 160, Loss: 0.0259, Acc: 1.0000
Epoch 180, Loss: 0.0225, Acc: 1.0000
Epoch 200, Loss: 0.0203, Acc: 1.0000
真实标签:
tensor([1, 1, 1, 1, 3, 3, 3, 1, 0, 1, 3, 1, 1, 1, 0, 0, 3, 1, 0, 1, 0, 1, 0, 0,
        2, 2, 0, 0, 2, 0, 0, 2, 0, 0])
预测标签:
tensor([1, 1, 1, 1, 3, 3, 3, 1, 0, 1, 3, 1, 1, 1, 0, 0, 3, 1, 0, 1, 0, 1, 0, 0,
        2, 2, 0, 0, 2, 0, 0, 2, 0, 0])
